# DiffusionGemma 画像入力検証 (Vision-Jev Guard)

カメラ画像 + プリセットの型付き設問 (Choice / Score / Noul) を DiffusionGemma に直接入力し、型付き JSON 判定が得られるかを確認します。

**ランタイムの選び方** (メニュー「ランタイム」→「ランタイムのタイプを変更」)

| モデル | 重み | 推奨 GPU |
|---|---|---|
| [1] 公式 BF16 | 51.6GB | A100 80GB / H100 |
| [2] FP8 | 27.2GB | H100 |
| [3] INT4 AWQ | 17.2GB | L4 (22.5GB) / A100 |

無料枠の T4 (16GB) では [3] も GPU に載り切りません (`--gpu-mem-gb 14` で CPU オフロードすれば低速ながら動作を試せます)。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/miyabiver39/VisionLocalJev.git
%cd VisionLocalJev
!pip install -q -r requirements-diffusiongemma.txt

## モデル選択 (1 行だけコメントアウトを外す)

In [ ]:
MODEL_ID = "trl-internal-testing/tiny-DiffusionGemmaForBlockDiffusion"  # [0] 動作確認用 (乱数重み・判定は無意味)
# MODEL_ID = "google/diffusiongemma-26B-A4B-it"                     # [1] BF16 51.6GB : A100 80GB / H100
# MODEL_ID = "RedHatAI/diffusiongemma-26B-A4B-it-FP8-dynamic"      # [2] FP8 27.2GB  : H100
# MODEL_ID = "cyankiwi/diffusiongemma-26B-A4B-it-AWQ-INT4"         # [3] INT4 17.2GB : L4 / A100

PRESET = "fire_disaster"   # security / fire_disaster / nursing_care / river_flood / factory_safety / railway_platform
EXTRA_ARGS = ""            # 例: "--gpu-mem-gb 14" (VRAM 不足時に CPU へオフロード)

## 画像のアップロード (任意)

スキップすると合成テスト画像 (炎のような橙色の塊) で判定します。

In [ ]:
IMAGE_PATH = ""
try:
    from google.colab import files
    up = files.upload()
    if up:
        IMAGE_PATH = next(iter(up))
except Exception as e:
    print("upload skipped:", e)
print("IMAGE_PATH =", IMAGE_PATH or "(synthetic)")

In [ ]:
img_arg = f"--image '{IMAGE_PATH}'" if IMAGE_PATH else ""
!python scripts/diffusiongemma_vision_check.py --model "$MODEL_ID" --preset "$PRESET" $img_arg $EXTRA_ARGS